# 🎨 Probador Interactivo de LoRAs SDXL: Ilustraciones de Libros Antiguos
### Fine-Tuning de Alta Resolución (1024×1024) sobre 
**Autor:** Twinber (Daniel Aguilera González)  
**Modelo Base:**   
**Repositorio en Hugging Face:** [](https://huggingface.co/Twinber/sdxl-oldbook-illustrations-lora)

---

## 📖 Descripción del Cuaderno
Este probador interactivo permite cargar y comparar directamente en una GPU gratuita de Google Colab (Tesla T4 de 15 GB) las **3 variantes de LoRA** entrenadas en RunPod sobre el dataset completo de 4.154 ilustraciones:
* **Variante A (=16, \alpha=16$):** Adaptación ligera de Cross-Attention (~47 MB).
* **Variante B (=32, \alpha=32$ - Recomendada):** Equilibrio óptimo entre estilización de grabado y fidelidad anatómica (~95 MB).
* **Variante C (=64, \alpha=32$):** Adaptación profunda con Cross-Attention + Proyecciones FFN (~189 MB).

Permite alternar entre los pesos de **1 época (500 pasos)** y **2 épocas (1.000 pasos)**, ajustar el peso del LoRA (*LoRA scale*) y generar imágenes a resolución nativa **1024×1024**.

## 1. Instalación de dependencias y configuración del entorno
Configuramos el entorno eliminando librerías conflictivas e instalando , ,  y .

In [ ]:
# Desinstalar posibles conflictos en el entorno de Colab
!pip uninstall -y -q torchao

# Instalar librerías del ecosistema Hugging Face
!pip install -q "diffusers>=0.30.0" "transformers>=4.44.0" "accelerate>=0.33.0" "peft>=0.12.0" "safetensors>=0.4.3" ipywidgets

## 2. Carga del Pipeline SDXL en FP16
Cargamos Stable Diffusion XL 1.0 en precisión media () con el scheduler ultra-rápido  (con Karras sigmas).

In [ ]:
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

# Escudo de compatibilidad para Colab
import diffusers.utils.import_utils
diffusers.utils.import_utils._torchao_available = False
diffusers.utils.import_utils.is_torchao_available = lambda: False

print("Cargando Stable Diffusion XL Base 1.0 en GPU...")
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

# Configurar scheduler rápido de 25-30 pasos
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)

# Habilitar optimizaciones de memoria
pipe.enable_vae_tiling()
print(f"¡Pipeline listo en GPU! VRAM en uso: {torch.cuda.memory_allocated() / (1024**3):.2f} GB / {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 3. Gestor de LoRAs desde Hugging Face Hub
Definimos el catálogo de adaptadores disponibles en el repositorio .

In [ ]:
REPO_ID = "Twinber/sdxl-oldbook-illustrations-lora"

LORA_WEIGHTS = {
    "Sin LoRA (SDXL Base 1.0)": None,
    "LoRA Variante A (r=16 - Final 1.000 pasos)": "sdxl_oldbook_lora_rank16_final.safetensors",
    "LoRA Variante B (r=32 - Final 1.000 pasos - RECOMENDADO)": "sdxl_oldbook_lora_rank32_final.safetensors",
    "LoRA Variante C (r=64 - Final 1.000 pasos)": "sdxl_oldbook_lora_rank64_final.safetensors",
    "LoRA Variante B (r=32 - Checkpoint 500 pasos / 1 época)": "checkpoints/variant_B_r32_step500.safetensors",
}

PRESET_PROMPTS = {
    "01. Botánica Exótica": "an antique botanical plate illustration of an exotic passionflower with leaves and roots, historical scientific drawing, detailed engraving, paper texture",
    "02. Relojero en Taller": "an antique portrait engraving of an elderly watchmaker with spectacles examining a pocket watch in his workshop, 19th century illustration",
    "03. Catedral Gótica y Mar": "a medieval gothic cathedral on a rocky cliff overlooking a stormy ocean, 19th century vintage landscape etching",
    "04. Locomotora en Viaducto": "a vintage 19th century steam locomotive crossing a stone viaduct over a deep mountain river, technical etching illustration",
    "05. Velero en Tempestad": "a tall wooden sailing ship battling gigantic stormy waves in the ocean, dark sky, antique vintage maritime engraving",
    "06. Coche Deportivo Moderno": "a modern red sports car parked on a wet asphalt city street at night, neon reflections, antique vintage etching",
    "07. Cafetería Contemporánea": "a young woman sitting in a modern coffee shop drinking a cup of coffee, looking out a large window, vintage book engraving style",
    "08. Golden Retriever en Colina": "a golden retriever dog sitting happily in a green grass field, bright sunny day, 19th century vintage illustration",
    "09. Rascacielos Futurista": "a futuristic glass skyscraper towering into the sky, urban metropolis with clouds, antique etching style",
    "10. Astronauta en Marte": "an astronaut walking on the red rocky surface of planet Mars, distant mountains, Earth in the sky, vintage scientific engraving",
    "-- Personalizado --": ""
}

current_lora = None

def apply_lora(lora_choice):
    global current_lora
    weight_file = LORA_WEIGHTS[lora_choice]
    if weight_file is None:
        if current_lora is not None:
            pipe.unload_lora_weights()
            current_lora = None
        return "Modelo Base (sin LoRA)"
    
    if current_lora != weight_file:
        pipe.unload_lora_weights()
        print(f"Descargando y aplicando {weight_file} desde {REPO_ID}...")
        pipe.load_lora_weights(REPO_ID, weight_name=weight_file)
        current_lora = weight_file
    return f"LoRA activo: {lora_choice}"

## 4. Interfaz Interactiva de Generación
Selecciona el modelo, el prompt predefinido (o escribe el tuyo propio), la semilla y el peso del LoRA.

In [ ]:
lora_dropdown = widgets.Dropdown(
    options=list(LORA_WEIGHTS.keys()),
    value="LoRA Variante B (r=32 - Final 1.000 pasos - RECOMENDADO)",
    description="Adaptador:",
    layout=widgets.Layout(width="70%")
)

prompt_dropdown = widgets.Dropdown(
    options=list(PRESET_PROMPTS.keys()),
    value="02. Relojero en Taller",
    description="Plantilla:",
    layout=widgets.Layout(width="70%")
)

prompt_text = widgets.Textarea(
    value=PRESET_PROMPTS["02. Relojero en Taller"],
    description="Prompt:",
    layout=widgets.Layout(width="90%", height="80px")
)

def on_prompt_change(change):
    if change["new"] != "-- Personalizado --":
        prompt_text.value = PRESET_PROMPTS[change["new"]]
prompt_dropdown.observe(on_prompt_change, names="value")

seed_input = widgets.IntText(value=42, description="Semilla:", layout=widgets.Layout(width="30%"))
steps_slider = widgets.IntSlider(value=30, min=15, max=50, step=5, description="Pasos:", layout=widgets.Layout(width="40%"))
cfg_slider = widgets.FloatSlider(value=7.5, min=4.0, max=12.0, step=0.5, description="CFG Scale:", layout=widgets.Layout(width="40%"))
scale_slider = widgets.FloatSlider(value=1.0, min=0.0, max=1.2, step=0.05, description="LoRA Scale:", layout=widgets.Layout(width="40%"))

generate_btn = widgets.Button(description="🚀 Generar Imagen 1024×1024", button_style="success", layout=widgets.Layout(width="50%", height="45px"))
output_area = widgets.Output()

def on_generate_clicked(b):
    with output_area:
        clear_output()
        status = apply_lora(lora_dropdown.value)
        print(status)
        print(f"Generando con semilla {seed_input.value}, {steps_slider.value} pasos, CFG {cfg_slider.value}...")
        
        generator = torch.Generator("cuda").manual_seed(seed_input.value)
        
        # Aplicar escala si hay LoRA activo
        cross_attention_kwargs = {"scale": scale_slider.value} if LORA_WEIGHTS[lora_dropdown.value] is not None else None
        
        image = pipe(
            prompt=prompt_text.value,
            num_inference_steps=steps_slider.value,
            guidance_scale=cfg_slider.value,
            generator=generator,
            cross_attention_kwargs=cross_attention_kwargs,
            width=1024,
            height=1024,
        ).images[0]
        
        filename = "resultado_generacion.png"
        image.save(filename)
        display(image)
        print(f"Imagen guardada como {filename} (1024×1024)")

generate_btn.on_click(on_generate_clicked)

display(widgets.VBox([
    lora_dropdown,
    prompt_dropdown,
    prompt_text,
    widgets.HBox([seed_input, steps_slider]),
    widgets.HBox([cfg_slider, scale_slider]),
    generate_btn,
    output_area
]))